# 01 — Create Lance Dataset from BDD100K

**Purpose:** Foundation notebook for the blueprint. Ingests BDD100K dashcam videos from Databricks Volumes, extracts frames as inline JPEG bytes using Ray GPU workers, and writes the result to a Lance table — the training-ready dataset consumed by `02_cnn_training.ipynb`.

---

## Why inline binary — not path references?

CNN training requires raw pixels on every batch. Storing a path string instead of image bytes introduces a second I/O hop: Lance read (fast) → object storage GET per image (slow, uncoalesced). At batch size 64 across 100 Ray workers, that's 6,400 concurrent object storage requests per training step. Per-request latency (~10–100ms) compounds and starves the GPU.

Inline binary co-locates pixels with metadata inside Lance fragments. A batch read becomes a single coalesced byte-range read within the fragment file — no per-image request overhead. Lance's blob layout stores heavy image payloads separately from structured metadata columns, so filtering on `video_id` or `timestamp_ms` never touches image bytes.

This is the core reason Lance is used here instead of Delta: Parquet row groups collapse from ~128K rows to ~1,280 rows at 100KB/image, making random batch access read 128MB to retrieve 1,280 samples. Lance's fragment-level O(1) addressing makes that cost constant regardless of dataset size.

---

## What this notebook does

1. **Configure paths.** Define the UC Volume path for raw BDD100K videos and the Lance dataset destination at `/Volumes/{catalog}/{schema}/{volume}/frames/`. Set frame sampling rate — 1fps yields ~4M frames across BDD100K's 100K videos; 3fps yields ~12M, crossing the threshold where Lance's random-access advantage over Parquet is measurable in training throughput.

2. **Define PyArrow schema.** Declare the frames table schema upfront so all Ray workers write consistent batches:
   - `frame_id` (string, UUID), `video_id` (string), `frame_number` (int32), `timestamp_ms` (int64)
   - `image` (binary — inline JPEG bytes), `width` / `height` (int32)
   - `bbox_labels` (list of structs: category, x1, y1, x2, y2) — BDD100K object detection annotations joined at write time
   - `extracted_at` (timestamp[us])

3. **Initialize Ray.** Start Ray on the Databricks cluster. Each worker is assigned a GPU for frame decoding.

4. **Extract frames per video (Ray worker).** Each worker receives a batch of video paths. For each video, `decord` decodes frames at the configured sampling rate, encodes each as JPEG bytes, and joins the BDD100K bounding box annotations for that video by `video_id`. Results are assembled into a PyArrow `RecordBatch`.

5. **Write to Lance.** Workers call `lance.write_dataset(batch, frames_path, mode='append', schema=schema)` directly — no Spark intermediate layer. Parallel appends across workers build independent Lance fragments, which Lance merges into a consistent dataset on completion.

6. **Verify.** Confirm row count and schema. Run `ds.take([0, 100_000, 5_000_000])` and time the result — random access should be constant regardless of which rows are requested, demonstrating Lance's O(1) guarantee.

---

**Inputs:** BDD100K video files in `/Volumes/{catalog}/{schema}/{volume}/videos/`; BDD100K annotation JSON (object detection labels)

**Outputs:** Lance `frames` dataset at `/Volumes/{catalog}/{schema}/{volume}/frames/` — inline JPEG bytes, metadata, and bounding box annotations, ready for CNN training

**Next:** `02_cnn_training.ipynb`